**Why CategoricalNB for Adult Income**

The Adult Income dataset contains mostly **categorical features** such as workclass, education, marital-status, occupation, relationship, race, sex, and native-country. **CategoricalNB** is designed specifically for categorical data and models the probability distribution of discrete feature values effectively.

**Why Naive Bayes Works Well**

* Handles **categorical and high-cardinality features** efficiently
* **Computationally fast** and scalable for large datasets
* **Less prone to overfitting** due to probabilistic modeling
* Performs well even with **limited training data**
* Simple model with **easy interpretation of probabilities**


**Dataset Loading and Inspection**

In [1]:
import pandas as pd

adult_data = pd.read_csv('drive/MyDrive/Datasets For ML/Adult_Income_Dataset.csv')

print("Columns:", adult_data.columns)
print("\nShape:", adult_data.shape)
print("\nDuplicated:", adult_data.duplicated().sum())
print("\nNull Values:", adult_data.isna().sum().sum())
print("\nFirst 2 Rows:")
adult_data.head(2)


Columns: Index(['age', 'workclass', 'fnlwgt', 'education', 'educational-num',
       'marital-status', 'occupation', 'relationship', 'race', 'gender',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country',
       'income'],
      dtype='object')

Shape: (48842, 15)

Duplicated: 52

Null Values: 0

First 2 Rows:


,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K


**Data Cleaning**

In [2]:
adult_data.drop_duplicates(inplace=True)
print("Duplicated Rows After Cleaning:", adult_data.duplicated().sum())

Duplicated Rows After Cleaning: 0


**Handling Missing Values**

Adult dataset usually has " ?" as missing values.

In [3]:
adult_data.replace(' ?', pd.NA, inplace=True)
adult_data.dropna(inplace=True)

print("Shape After Removing Missing Values:", adult_data.shape)

Shape After Removing Missing Values: (48790, 15)


**Feature–Label Separation**

In [4]:
X, Y = adult_data.drop(columns=['income']), adult_data['income']

**Selecting Only Categorical Features**

CategoricalNB requires integer-encoded categorical values.

In [5]:
categorical_features = X.select_dtypes(include=['object']).columns
X_cat = X[categorical_features]

print("Categorical Features:\n", categorical_features)

Categorical Features:
 Index(['workclass', 'education', 'marital-status', 'occupation',
       'relationship', 'race', 'gender', 'native-country'],
      dtype='object')


**Encoding Categorical Features (Label Encoding)**

Each feature must be encoded independently.

In [6]:
from sklearn.preprocessing import LabelEncoder

encoders = {}
X_encoded = X_cat.copy()

for col in X_cat.columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_cat[col])
    encoders[col] = le

# Encode target variable
target_encoder = LabelEncoder()
Y_encoded = target_encoder.fit_transform(Y)

**Train–Test Split (Stratified)**

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X_encoded, Y_encoded, test_size=0.2, stratify=Y_encoded, random_state=5)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (39032, 8)
Test Shape: (9758, 8)


**Implementing Categorical Naive Bayes**

In [8]:
from sklearn.naive_bayes import CategoricalNB

cnb = CategoricalNB(alpha=1.0)
cnb.fit(X_train, Y_train)

CategoricalNB()

**Model Predictions and Probabilities**

In [9]:
Y_pred = cnb.predict(X_test)
print("Predicted Labels:\n", Y_pred)

print("\nPredicted Probabilities:\n", cnb.predict_proba(X_test))

Predicted Labels:
 [1 0 0 ... 0 0 1]

Predicted Probabilities:
 [[0.01214985 0.98785015]
 [0.97266248 0.02733752]
 [0.91468036 0.08531964]
 ...
 [0.54248013 0.45751987]
 [0.96968936 0.03031064]
 [0.01586504 0.98413496]]


**Model Evaluation Metrics**

In [13]:
from sklearn import metrics

print("Accuracy Score:", metrics.accuracy_score(Y_test, Y_pred))
print("Precision Score:", metrics.precision_score(Y_test, Y_pred))
print("Recall Score:", metrics.recall_score(Y_test, Y_pred))
print("F1 Score:", metrics.f1_score(Y_test, Y_pred))
print("\nClassification Report:\n", metrics.classification_report(Y_test, Y_pred, target_names=target_encoder.classes_))
print("Confusion Matrix:\n", metrics.confusion_matrix(Y_test, Y_pred))

Accuracy Score: 0.7982168477146956
Precision Score: 0.5594428247489472
Recall Score: 0.7392979452054794
F1 Score: 0.6369168356997972

Classification Report:
               precision    recall  f1-score   support

       <=50K       0.91      0.82      0.86      7422
        >50K       0.56      0.74      0.64      2336

    accuracy                           0.80      9758
   macro avg       0.73      0.78      0.75      9758
weighted avg       0.83      0.80      0.81      9758

Confusion Matrix:
 [[6062 1360]
 [ 609 1727]]


**Exploring Model Parameters**

In [11]:
print("Model Parameters:", cnb.get_params())
print("Class Log Prior:", cnb.class_log_prior_)

Model Parameters: {'alpha': 1.0, 'class_prior': None, 'fit_prior': True, 'force_alpha': True, 'min_categories': None}
Class Log Prior: [-0.27367258 -1.42954038]


**Predicting Income for a New Sample**

In [12]:
sample = {
    'workclass': 'Private',
    'education': 'Bachelors',
    'marital-status': 'Never-married',
    'occupation': 'Tech-support',
    'relationship': 'Not-in-family',
    'race': 'White',
    'gender': 'Male',
    'native-country': 'United-States'
}

sample_df = pd.DataFrame([sample])

for col in sample_df.columns:
    sample_df[col] = encoders[col].transform(sample_df[col])
prediction = cnb.predict(sample_df)
print("Predicted Income Class:", target_encoder.inverse_transform(prediction))

Predicted Income Class: ['<=50K']


**Advantages of CategoricalNB for Adult Income**

No scaling required

Handles many categories efficiently

Lower overfitting than Decision Trees

Faster training and inference


**Limitations**

Assumes conditional independence

Ignores numerical features unless discretized

Probabilities may be poorly calibrated

**Conclusion**

The Categorical Naive Bayes model was implemented on the Adult Income dataset using encoded categorical features. After data cleaning, preprocessing, and training, the model was evaluated using accuracy, precision, recall, and F1-score. The results show that CategoricalNB efficiently handles categorical data and provides fast and effective classification for predicting income categories, although it relies on the assumption of feature independence.
